# 06 — Neurons, layers, and MLPs

Part of the micrograd repetition pack.


## Goal
Compose scalar `Value` objects into neural-network modules and reason about shapes and parameter counts.


In [ ]:
import math

_results = []

def check(name, condition, detail=""):
    ok = bool(condition)
    _results.append(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f" — {detail}" if detail else ""))

def close(a, b, tol=1e-6):
    return abs(a - b) <= tol

def summary():
    print(f"\nScore: {sum(_results)}/{len(_results)} tests passed")


In [ ]:
from math import exp, log, sin, cos

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float))
        out = Value(self.data ** power, (self,), f'**{power}')
        def _backward():
            self.grad += power * self.data ** (power - 1) * out.grad
        out._backward = _backward
        return out

    def sin(self):
        out = Value(sin(self.data), (self,), 'sin')
        def _backward():
            self.grad += cos(self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        e2x = (2 * self).exp()
        return (e2x - 1) / (e2x + 1)

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1
    def __rtruediv__(self, other): return Value(other) * self ** -1

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [ ]:
import random
random.seed(42)


In [ ]:
class Module:
    def parameters(self):
        return []
    def zero_grad(self):
        # TODO
        pass

class Neuron(Module):
    def __init__(self, nin):
        # TODO: nin weights and one bias
        pass
    def __call__(self, x):
        # TODO: validate input length, weighted sum + bias, tanh
        raise NotImplementedError
    def parameters(self):
        # TODO
        raise NotImplementedError

class Layer(Module):
    def __init__(self, nin, nout):
        # TODO
        pass
    def __call__(self, x):
        # TODO: return a scalar only for one-output layers
        raise NotImplementedError
    def parameters(self):
        # TODO: flatten neuron parameter lists
        raise NotImplementedError

class MLP(Module):
    def __init__(self, nin, nouts):
        # TODO: pair [nin] + nouts into layers
        pass
    def __call__(self, x):
        # TODO
        raise NotImplementedError
    def parameters(self):
        # TODO
        raise NotImplementedError


### Round A — Neuron


In [ ]:
_results.clear()
try:
    n=Neuron(3)
    check("three weights",len(n.w)==3)
    check("four neuron parameters",len(n.parameters())==4)
    y=n([2,-1,0.5])
    check("neuron returns Value",isinstance(y,Value))
    check("tanh range",-1<y.data<1)
except Exception as e: check("Neuron runs",False,repr(e))
summary()


### Round B — Layer and MLP
Before running, calculate the parameter count of `MLP(3,[4,4,1])` by hand.


In [ ]:
_results.clear()
try:
    layer=Layer(3,4); out=layer([1,2,3])
    check("four outputs",len(out)==4)
    check("layer parameters",len(layer.parameters())==16)
    mlp=MLP(3,[4,4,1])
    check("three layers",len(mlp.layers)==3)
    check("MLP parameter count",len(mlp.parameters())==41)
    check("final output is scalar Value",isinstance(mlp([1,2,3]),Value))
except Exception as e: check("Layer/MLP runs",False,repr(e))
summary()


### Round C — Debug a silent shape bug
The source notebook created `MLP(3, ...)` but also showed an input with four entries. Python's `zip` silently drops the extra entry. Add explicit validation so this raises `ValueError`.


In [ ]:
_results.clear()
try:
    mlp=MLP(3,[4,4,1])
    raised=False
    try: mlp([0,3,-2,1])
    except ValueError: raised=True
    check("wrong input width is rejected",raised)
except Exception as e: check("shape validation test",False,repr(e))

try:
    mlp=MLP(3,[4,4,1])
    for p in mlp.parameters(): p.grad=7
    mlp.zero_grad()
    check("zero_grad clears every parameter",all(p.grad==0 for p in mlp.parameters()))
except Exception as e: check("zero_grad runs",False,repr(e))
summary()


### Concept checks

1. Why does a neuron with `nin` inputs have `nin + 1` parameters?
2. Derive the 41-parameter count for `[3,4,4,1]`.
3. Track the data shape through every layer.
4. Rebuild all four classes from memory.
